In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
pd.set_option('display.max_columns', None)
import copy
import os
import sklearn
from sklearn.model_selection import train_test_split
from pcntoolkit import NormData
import warnings
import logging
import pcntoolkit.util.output
from PIL import Image
from pcntoolkit import (
    BLR,
    BsplineBasisFunction,
    NormativeModel,
    NormData,
    load_fcon1000,
    plot_centiles_advanced,
    plot_qq,
    plot_ridge,
)


sns.set_style("darkgrid")

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")

pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
pcntoolkit.util.output.Output.set_show_messages(True)


#this code was needed to run the code below without errors
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    # Legacy Python that doesn't verify HTTPS certificates by default
    pass
else:
    # Handle target environment that doesn't support HTTPS verification
    ssl._create_default_https_context = _create_unverified_https_context

print('test')
pd.set_option('display.max_rows', 150)

test


In [4]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data"

euler_nr = pd.read_csv(os.path.join(path, "euler_nr_data.csv"))
datatable = pd.read_csv(os.path.join(path, "Final_data.csv"))

euler_nr = euler_nr[['SubjectID', 'avg_euler_centered_neg_sqrt']].copy()
df_ukb = datatable.merge(euler_nr, how="right")
df_ukb = df_ukb.iloc[:, 1:]
df_ukb = df_ukb.dropna(axis=1)

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df_ukb["avg_euler_centered_neg_sqrt"] = df_ukb["avg_euler_centered_neg_sqrt"].astype(float)
df_ukb = df_ukb[df_ukb["avg_euler_centered_neg_sqrt"] < 10.0]

df_ukb = df_ukb.drop(columns=['SubjectID','Year_initial_scan','Birthyear','avg_euler_centered_neg_sqrt','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities', 'non-WM-hypointensities'])
df_ukb["Dataset"]= "UKB"
first_cols = ["Age", "Sex", "Site", "Dataset"]
other_cols = df_ukb.columns.difference(first_cols, sort=False)
df_ukb = df_ukb[first_cols + list(other_cols)]

df_o3 = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/Oasis3_data.csv")

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df_o3 = df_o3.drop(columns=['SubjectID','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities'])

pd.set_option('display.max_columns', None)
df_o3 = df_o3.iloc[:,1:]

#split datasets into alzheimer vs healthy controls
alzheimer_df = df_o3[df_o3["Diagnosis"] >= 1.0]
healthy_controls_o3 = df_o3[df_o3["Diagnosis"] == 0.0]

#remove diagnosis from the Healthy control dataframe
healthy_controls_o3 = healthy_controls_o3.drop(columns=['Diagnosis'])

#order the HC the same way as the UKB dataset
ordering_ukb = list(df_ukb.columns)
healthy_controls_o3["Dataset"] = "OASIS3"
healthy_controls_o3 = healthy_controls_o3[ordering_ukb]
df_o3["Dataset"] = "OASIS3"

df_adni = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/data_prep/ADNI_data.csv")
df_adni = df_adni.iloc[:,1:]
df_adni.replace(to_replace=['F'], value=0.0, inplace=True)
df_adni.replace(to_replace=['M'], value=1.0, inplace=True)
df_adni = df_adni[df_adni["Sex"] != "X"]
df_adni["Age"] = df_adni["Age"].round()
healthy_controls_adni = df_adni[df_adni["Group"] == "CN"]
healthy_controls_adni.drop(columns=["Group"])
healthy_controls_adni["Dataset"] = "ADNI"
healthy_controls_adni = healthy_controls_adni[ordering_ukb]
df_adni["Dataset"] = "ADNI"

data_df = pd.concat([df_ukb, healthy_controls_o3, healthy_controls_adni], axis=0)
data_df = data_df[~data_df["Site"].isin([9.0, 10.0, 14.0, 16.0, 19.0, 20.0, 24.0, 27.0, 35.0, 51.0, 62.0, 67.0, 70.0, 114.0, 130.0, 133.0, 135.0, 136.0, 141.0, 153.0, 57.0, 68.0])]

#removing these ages because they had unique site & age combinations which could not be split 
data_df = data_df[data_df["Age"] != 96.0]
data_df = data_df[data_df["Age"] != 91.0]
data_df = data_df[data_df["Age"] != 95.0]
data_df = data_df[data_df["Age"] != 97.0]

data = data_df.copy()
covariates = ["Age"]
batch_effects = ["Sex", "Site"]
data_df
data_df = data_df.drop(columns=['Age','Sex','Site'])
columns = list(data_df.columns)

response_vars = columns

# create a NormData object
norm_data = NormData.from_dataframe(
    name="Final_BLR_corrected", dataframe=data, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars
)
norm_data.coords

train_plt, test_plt = train_test_split(data, test_size=0.2, random_state=42)
train, test = norm_data.train_test_split([0.8, 0.2], random_state=42)

/scratch/quirom/slurm_job_53613890/ipykernel_468393/509808752.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ukb["Dataset"]= "UKB"
/scratch/quirom/slurm_job_53613890/ipykernel_468393/509808752.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  healthy_controls_o3["Dataset"] = "OASIS3"
/scratch/quirom/slurm_job_53613890/ipykernel_468393/509808752.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all 

Process: 468393 - 2026-06-12 16:21:30 - Dataset "Final_BLR_corrected" created.
    - 35378 observations
    - 35378 unique subjects
    - 1 covariates
    - 104 response variables
    - 2 batch effects:
    	Sex (2)
	Site (35)
    


KeyError: "No variable named 'Observations'. Did you mean one of ('observations',)?"